# torch.masked — First-Class Missing Data in PyTorch

**Module 24** of the PyTorch Complete Learning Guide

This notebook explores the `torch.masked` module and `MaskedTensor` — PyTorch's approach to handling missing and invalid data through first-class boolean masks.

**Topics covered:**
- The masking problem (padding in NLP, missing data in tabular)
- Creating MaskedTensors
- Masked reductions: correct mean ignoring padding
- Masked softmax for attention
- Padded sequence example: comparing 3 approaches
- `torch.masked.*` function API
- Mask propagation semantics
- Limitations and when to use

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.masked import MaskedTensor

print(f"PyTorch version: {torch.__version__}")
torch.manual_seed(42)

## 1. The Masking Problem

Missing data is everywhere in deep learning:

| Domain | Scenario | What's "Missing" |
|--------|----------|------------------|
| **NLP** | Padded sequences | Positions beyond true length |
| **Vision** | Irregular shapes | Pixels outside ROI |
| **Tabular** | Incomplete records | Unobserved columns |
| **Attention** | Causal/padding masks | Future or padding tokens |

Let's see why naive approaches fail.

In [ ]:
# Three sequences of different lengths, padded to max_len=5
seq_a = torch.tensor([3.0, 1.0, 4.0])        # length 3
seq_b = torch.tensor([2.0, 7.0])              # length 2
seq_c = torch.tensor([5.0, 3.0, 2.0, 8.0])   # length 4

padded = nn.utils.rnn.pad_sequence(
    [seq_a, seq_b, seq_c], batch_first=True, padding_value=0.0
)
lengths = torch.tensor([3, 2, 4])

print("Padded batch:")
print(padded)
print(f"\nNaive mean (WRONG — includes padding zeros):")
print(f"  {padded.mean(dim=1)}")
print(f"\nTrue means:")
print(f"  [{seq_a.mean():.4f}, {seq_b.mean():.4f}, {seq_c.mean():.4f}]")

The naive mean divides by 4 (max_len) instead of each sequence's actual length. Padding zeros dilute the result.

## 2. Creating MaskedTensors

`MaskedTensor` pairs a data tensor with a boolean mask tensor of the same shape.

Convention: **`True` = valid**, **`False` = masked/missing**.

In [ ]:
# 1D MaskedTensor
data = torch.tensor([10.0, 20.0, 30.0, 0.0, 0.0])
mask = torch.tensor([True, True, True, False, False])
mt = MaskedTensor(data, mask)
print("1D MaskedTensor:")
print(mt)

In [ ]:
# 2D MaskedTensor — batch of padded sequences
data_2d = torch.tensor([
    [1.0, 2.0, 3.0, 0.0, 0.0],
    [4.0, 5.0, 0.0, 0.0, 0.0],
    [6.0, 7.0, 8.0, 9.0, 0.0],
])
lengths = torch.tensor([3, 2, 4])
mask_2d = torch.arange(5).unsqueeze(0) < lengths.unsqueeze(1)

mt_2d = MaskedTensor(data_2d, mask_2d)
print("2D MaskedTensor:")
print(mt_2d)
print(f"\nMask:\n{mask_2d}")

In [ ]:
# Accessing data and mask
print(f"get_data(): {mt.get_data()}")
print(f"get_mask(): {mt.get_mask()}")
print(f"dtype: {mt.dtype}")
print(f"shape: {mt.shape}")

## 3. Masked Reductions: Correct Mean Ignoring Padding

The `torch.masked._ops` module provides reductions that correctly ignore masked elements.

In [ ]:
data = torch.tensor([
    [3.0, 1.0, 4.0, 0.0, 0.0],
    [2.0, 7.0, 0.0, 0.0, 0.0],
    [5.0, 3.0, 2.0, 8.0, 0.0],
])
lengths = torch.tensor([3, 2, 4])
mask = torch.arange(5).unsqueeze(0) < lengths.unsqueeze(1)

print("Regular mean (WRONG):")
print(f"  {data.mean(dim=1)}")

print("\nMasked mean (CORRECT):")
masked_mean = torch.masked._ops.mean(data, dim=1, mask=mask)
print(f"  {masked_mean}")

print("\nMasked sum:")
masked_sum = torch.masked._ops.sum(data, dim=1, mask=mask)
print(f"  {masked_sum}")

In [ ]:
# amax and amin — ignore masked junk values
data_with_junk = torch.tensor([
    [3.0, 1.0, 4.0, 99.0, 99.0],
    [2.0, 7.0, -5.0, -5.0, -5.0],
])
mask2 = torch.tensor([
    [True, True, True, False, False],
    [True, True, False, False, False],
])

print(f"Regular max: {data_with_junk.amax(dim=1)}  (includes junk 99.0!)")
print(f"Masked max:  {torch.masked._ops.amax(data_with_junk, dim=1, mask=mask2)}")
print(f"\nRegular min: {data_with_junk.amin(dim=1)}  (includes junk -5.0!)")
print(f"Masked min:  {torch.masked._ops.amin(data_with_junk, dim=1, mask=mask2)}")

## 4. Masked Softmax for Attention

The most common use case: computing softmax where some positions should be ignored (padding, future tokens).

`torch.masked.softmax` normalizes over **valid elements only**.

In [ ]:
scores = torch.tensor([
    [0.5, 1.2, 0.3, 0.0, 0.0],
    [2.0, 1.0, 3.0, 0.5, 0.0],
])
mask = torch.tensor([
    [True, True, True, False, False],
    [True, True, True, True, False],
])

# Manual approach: masked_fill + softmax
manual = torch.softmax(scores.masked_fill(~mask, float('-inf')), dim=1)
print("Manual masked_fill + softmax:")
print(f"  {manual}")

# torch.masked.softmax
masked = torch.masked.softmax(scores, dim=1, mask=mask)
print("\ntorch.masked.softmax:")
print(f"  {masked}")

print(f"\nRow 0 valid probabilities sum: {masked[0, :3].sum():.4f}")
print(f"Row 1 valid probabilities sum: {masked[1, :4].sum():.4f}")

In [ ]:
# Edge case: all positions masked
all_masked = torch.tensor([[False, False, False]])
scores_edge = torch.tensor([[1.0, 2.0, 3.0]])

manual_edge = torch.softmax(
    scores_edge.masked_fill(~all_masked, float('-inf')), dim=1
)
safe_edge = torch.masked.softmax(scores_edge, dim=1, mask=all_masked)

print(f"All-masked row:")
print(f"  Manual: {manual_edge}  (NaN!)")
print(f"  Masked: {safe_edge}  (safe zeros)")

## 5. Padded Sequence Mean: Comparing 3 Approaches

A side-by-side comparison of computing the true mean of variable-length sequences.

In [ ]:
# Setup
seq_a = torch.tensor([3.0, 1.0, 4.0])
seq_b = torch.tensor([2.0, 7.0])
seq_c = torch.tensor([5.0, 3.0, 2.0, 8.0])

padded = nn.utils.rnn.pad_sequence(
    [seq_a, seq_b, seq_c], batch_first=True
)
lengths = torch.tensor([3, 2, 4])
mask = torch.arange(padded.size(1)).unsqueeze(0) < lengths.unsqueeze(1)

true_means = torch.tensor([
    seq_a.mean().item(),
    seq_b.mean().item(),
    seq_c.mean().item(),
])

print(f"Padded data:\n{padded}")
print(f"Lengths: {lengths}")
print(f"True means: {true_means}")

In [ ]:
# Approach 1: Naive (WRONG)
naive = padded.mean(dim=1)
print(f"Approach 1 — Naive:   {naive}  <-- WRONG")

# Approach 2: Manual masking (correct but verbose)
manual = (padded * mask.float()).sum(dim=1) / mask.float().sum(dim=1)
print(f"Approach 2 — Manual:  {manual}")

# Approach 3: torch.masked API (clean)
api = torch.masked._ops.mean(padded, dim=1, mask=mask)
print(f"Approach 3 — Masked:  {api}")

print(f"\nApproach 2 correct: {torch.allclose(manual, true_means)}")
print(f"Approach 3 correct: {torch.allclose(api, true_means)}")

## 6. `torch.masked.*` Function API

The `torch.masked` module provides these functions that work on **plain tensors** with an explicit mask argument:

| Function | Description |
|----------|-------------|
| `torch.masked._ops.sum` | Sum of valid elements |
| `torch.masked._ops.mean` | Mean of valid elements |
| `torch.masked._ops.amax` | Max of valid elements |
| `torch.masked._ops.amin` | Min of valid elements |
| `torch.masked._ops.prod` | Product of valid elements |
| `torch.masked._ops.var` | Variance of valid elements |
| `torch.masked._ops.std` | Std deviation of valid elements |
| `torch.masked.softmax` | Softmax over valid elements |
| `torch.masked.log_softmax` | Log-softmax over valid elements |
| `torch.masked.normalize` | Normalize over valid elements |

In [ ]:
data = torch.tensor([[1.0, 2.0, 3.0, 0.0],
                      [4.0, 5.0, 0.0, 0.0]])
mask = torch.tensor([[True, True, True, False],
                      [True, True, False, False]])

print(f"Data:\n{data}")
print(f"Mask:\n{mask}")
print()

print(f"sum:  {torch.masked._ops.sum(data, dim=1, mask=mask)}")
print(f"mean: {torch.masked._ops.mean(data, dim=1, mask=mask)}")
print(f"amax: {torch.masked._ops.amax(data, dim=1, mask=mask)}")
print(f"amin: {torch.masked._ops.amin(data, dim=1, mask=mask)}")
print(f"prod: {torch.masked._ops.prod(data, dim=1, mask=mask)}")

In [ ]:
# Masked log_softmax
log_sm = torch.masked.log_softmax(data, dim=1, mask=mask)
print(f"log_softmax:\n{log_sm}")

# Masked normalize (L2)
normed = torch.masked.normalize(data, ord=2.0, dim=1, mask=mask)
print(f"\nnormalize (L2):\n{normed}")

## 7. Mask Propagation Semantics

When using `MaskedTensor`, operations follow specific rules:

| Operation Type | Mask Rule | Example |
|---------------|-----------|--------|
| **Unary** | Preserve mask | `abs(mt)` keeps the same mask |
| **Binary** | AND masks | `mt_a + mt_b` → valid only if both valid |
| **Reduction** | Collapse | `mt.sum(dim)` → valid if any element in dim was valid |

In [ ]:
# Unary: mask preserved
mt = MaskedTensor(
    torch.tensor([1.0, -2.0, 3.0, 0.0]),
    torch.tensor([True, True, True, False])
)
print(f"Original:  {mt}")
print(f"abs():     {mt.abs()}")
print(f"neg():     {(-mt)}")

In [ ]:
# Binary: masks ANDed
a = MaskedTensor(
    torch.tensor([1.0, 2.0, 3.0]),
    torch.tensor([True, True, False])
)
b = MaskedTensor(
    torch.tensor([10.0, 20.0, 30.0]),
    torch.tensor([True, False, True])
)

result = a + b
print(f"a:     {a}")
print(f"b:     {b}")
print(f"a + b: {result}")
print(f"\nMask a: {a.get_mask().tolist()}")
print(f"Mask b: {b.get_mask().tolist()}")
print(f"Result: {(a.get_mask() & b.get_mask()).tolist()}")

In [ ]:
# Reduction: mask collapses
mt_2d = MaskedTensor(
    torch.tensor([[1.0, 2.0, 0.0],
                  [4.0, 0.0, 0.0]]),
    torch.tensor([[True, True, False],
                  [True, False, False]])
)
print(f"2D MaskedTensor:\n{mt_2d}")
print(f"\nsum(dim=1): {mt_2d.sum(dim=1)}")

## 8. Practical: Masked Attention Scores

Using `torch.masked.softmax` for attention with padding masks.

In [ ]:
torch.manual_seed(42)
batch, seq_len, d_k = 2, 5, 4
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

lengths = torch.tensor([3, 5])
key_mask = torch.arange(seq_len).unsqueeze(0) < lengths.unsqueeze(1)
attn_mask = key_mask.unsqueeze(1).expand(-1, seq_len, -1)

scores = (Q @ K.transpose(-2, -1)) / (d_k ** 0.5)

attn_weights = torch.masked.softmax(scores, dim=-1, mask=attn_mask)
output = attn_weights @ V

print(f"Attention scores shape: {scores.shape}")
print(f"Key mask: {key_mask}")
print(f"\nAttention weights (batch 0):\n{attn_weights[0]}")
print(f"\nRow sums (batch 0): {attn_weights[0].sum(dim=-1)}")
print(f"Output shape: {output.shape}")

Notice that for batch 0, only the first 3 key positions have nonzero attention weights. The row sums are 1.0 (properly normalized over valid positions only).

## 9. Limitations and When to Use

**MaskedTensor is a prototype feature.** Not all operations are supported.

| Aspect | Status |
|--------|--------|
| Basic arithmetic | Supported |
| Unary ops (abs, neg, exp) | Supported |
| Reductions (sum, mean, max) | Supported |
| Masked softmax | Supported |
| nn.functional ops | Partial |
| torch.compile | Partial |
| All shape ops | Partial |
| Gradient through mask | Not supported (mask is fixed) |

In [ ]:
mt = MaskedTensor(
    torch.tensor([1.0, 2.0, 3.0, 0.0]),
    torch.tensor([True, True, True, False])
)

# Supported ops
for op_name in ['abs', 'neg', 'sum', 'mean']:
    try:
        getattr(mt, op_name)()
        print(f"  {op_name}: supported")
    except Exception as e:
        print(f"  {op_name}: NOT supported — {str(e)[:60]}")

# Potentially unsupported
for name, fn in [('sort', lambda: torch.sort(mt)),
                  ('reshape', lambda: mt.reshape(2, 2))]:
    try:
        fn()
        print(f"  {name}: supported")
    except Exception as e:
        print(f"  {name}: NOT supported — {str(e)[:60]}")

### When to use which approach

| Scenario | Recommendation |
|----------|---------------|
| Masked reductions (sum, mean, max) | `torch.masked._ops.*` |
| Masked softmax in attention | `torch.masked.softmax` |
| Automatic mask propagation in prototyping | `MaskedTensor` |
| Performance-critical production code | Manual masking |
| Complex multi-mask scenarios | Manual masking |
| Operations not yet supported | Manual masking |

## Exercise: Compute Masked Cross-Entropy Loss for Variable-Length Sequences

**Task:** Given a batch of predictions and targets with different sequence lengths, compute the mean cross-entropy loss ignoring padding positions.

```
# Setup:
batch_size, max_len, vocab_size = 3, 6, 10
logits = torch.randn(batch_size, max_len, vocab_size)
targets = torch.randint(0, vocab_size, (batch_size, max_len))
lengths = torch.tensor([4, 6, 3])

# Your task:
# 1. Compute per-position cross-entropy loss (no reduction)
# 2. Create a mask from lengths
# 3. Zero out losses at padding positions
# 4. Compute the mean loss over valid positions only
# 5. Compare: naive mean vs correct masked mean
```

In [ ]:
# Exercise solution
torch.manual_seed(42)
batch_size, max_len, vocab_size = 3, 6, 10
logits = torch.randn(batch_size, max_len, vocab_size)
targets = torch.randint(0, vocab_size, (batch_size, max_len))
lengths = torch.tensor([4, 6, 3])

# Step 1: per-position cross-entropy (no reduction)
per_pos_loss = F.cross_entropy(
    logits.view(-1, vocab_size), targets.view(-1), reduction='none'
).view(batch_size, max_len)

# Step 2: mask from lengths
mask = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)

# Step 3 & 4: masked mean using torch.masked API
masked_loss = torch.masked._ops.mean(per_pos_loss, dim=1, mask=mask)

# Comparison
naive_loss = per_pos_loss.mean(dim=1)

print(f"Per-position loss shape: {per_pos_loss.shape}")
print(f"Lengths: {lengths}")
print(f"\nNaive mean loss per sequence:  {naive_loss}")
print(f"Masked mean loss per sequence: {masked_loss}")
print(f"\nOverall naive mean:  {naive_loss.mean():.4f}")
print(f"Overall masked mean: {masked_loss.mean():.4f}")

## Key Takeaways

1. **Missing data is everywhere** — padding, irregular shapes, missing values. Manual masking is verbose and error-prone.

2. **`MaskedTensor`** pairs data + boolean mask. Operations respect the mask automatically. `True` = valid, `False` = masked.

3. **`torch.masked.softmax`** normalizes over valid elements only — safer than `masked_fill(-inf)` + softmax.

4. **`torch.masked._ops.{sum, mean, amax, amin, prod}`** — correct reductions that divide by valid count, not total count.

5. **Mask propagation**: unary ops preserve mask, binary ops AND masks, reductions collapse masks.

6. **Prototype status**: not all ops supported, potential performance overhead. Use `torch.masked.*` functions for reductions/softmax; use manual masking for unsupported or performance-critical paths.

7. **The `torch.masked.*` standalone functions** (softmax, log_softmax, normalize, reductions) work on **plain tensors** — you don't need `MaskedTensor` to benefit from them.